# Lab 3 — Watch the Actor Drive

![Acting: observation to state estimation, Actor, sampled action and real environment, with observation feedback](figures/acting.svg)

**Used to act:** Encoder, RSSM state estimation and Actor. **Not used to select an action:** Critic, reward/continuation heads, imagined training rollouts and optimizer. The checkpoint supplies trained parameters; Lab 3 does not train or update them.

## SETTINGS — EDIT HERE

Train a policy in Lab 4, copy its generated `CHECKPOINT` line into the settings below,
and rerun Lab 3 to watch your policy drive. Changing `CHECKPOINT` requires rerunning inference with **Kernel → Restart Kernel and Run All Cells**; replaying captured video does not load a new policy. Use the complete run directory
so its training and reward configuration can be loaded; only load trusted
runs because checkpoint files contain executable Python pickle data.

`SEED` controls the simulator's starting traffic and the Actor's sampling randomness; Lab 4's `SEED` controls training, and the two need not match. Compare policies using the same Lab 3 seed for a similar initial scenario. Because actions are sampled and change future traffic observations, matching seeds does not guarantee identical trajectories.

Lab 3 loads trained parameters for the Encoder, RSSM and Actor: the Encoder and RSSM estimate the current state, and the Actor chooses an action. The Critic is used during training, not to choose actions while driving.

### Examples — Try These

These are examples, not executable cells. Edit only the settings cell below.

**1. Watch the provided Actor**
```python
CHECKPOINT = None
SEED = 1000
```
**2. Watch your Lab 4 Actor**
```python
CHECKPOINT = "~/dreamer-labs-3-4/results/my_driver-d9bb72ea"
SEED = 1000
```
This path is only an example, not a guaranteed path. Paste the actual `CHECKPOINT` assignment printed at the end of your Lab 4 run.

**3. Try a different driving scenario**
```python
CHECKPOINT = None
SEED = 1001
```

In [1]:
# SETTINGS — EDIT HERE
CHECKPOINT = None
SEED = 1000

A trained DreamerV3 agent drives a real HighwayEnv episode, live. At every
decision, once per simulated second:

1. HighwayEnv gives the current **5×5 Kinematics observation**: positions and
   velocities of the agent's car and up to four nearby cars.
2. DreamerV3's **Encoder + RSSM** update the agent's internal state from it.
3. The trained **Actor** gives a probability for each of the five actions:
   LEFT, IDLE, RIGHT, FASTER, SLOWER.
4. One action is **sampled** from those probabilities.
5. HighwayEnv **executes** the sampled action, and the cycle repeats.

This is real model inference and real simulation, not a recording. Nothing is
trained.

Follow `START_HERE.md` to install and launch JupyterLab with the shared
environment. Select **Python 3 (ipykernel)** if prompted.
Use **Kernel → Restart Kernel and Run All Cells**. The initial scene appears
first; the agent loads and driving starts automatically.

In [2]:
import os, sys
from pathlib import Path

ROOT = Path(sys.prefix).resolve().parent
if (sys.version_info[:3] != (3, 11, 12) or
        Path(sys.prefix).resolve() != ROOT / '.venv' or
        not os.environ.get('VIRTUAL_ENV') or
        Path(os.environ['VIRTUAL_ENV']).resolve() != Path(sys.prefix).resolve() or
        not (ROOT / 'pyproject.toml').is_file() or
        not (ROOT / 'inference' / 'lab3_inference.py').is_file() or
        not (ROOT / 'optional_training' / 'train.py').is_file()):
    raise RuntimeError('Wrong kernel. From the extracted lab folder run uv sync --locked, then uv run --locked jupyter lab; select Python 3 (ipykernel) and restart the kernel.')

runtime = ROOT
for name, module in tuple(sys.modules.items()):
    if name.split('.')[0] not in ('dreamerv3', 'embodied', 'optional_training'):
        continue
    locations = list(getattr(module, '__path__', ()))
    if getattr(module, '__file__', None):
        locations.append(module.__file__)
    expected_package = runtime / name.split('.')[0]
    if not locations or any(not Path(p).resolve().is_relative_to(expected_package) for p in locations):
        raise RuntimeError(f'{name} came from another runtime. Restart this notebook kernel before running it.')
for name in ('lab3_inference', 'live_view'):
    if name in sys.modules and (not getattr(sys.modules[name], '__file__', None) or
            Path(sys.modules[name].__file__).resolve() != ROOT / 'inference' / f'{name}.py'):
        raise RuntimeError(f'{name} came from another runtime. Restart this notebook kernel before running it.')
sys.path.insert(0, str(runtime))

sys.path.insert(0, str(ROOT / "inference"))

from lab3_inference import LiveDrive
from live_view import LiveDriveView

if "view" in globals():
    view.close()
    drive.close()
run = None if CHECKPOINT is None else ROOT / Path(CHECKPOINT).expanduser()
drive = LiveDrive(seed=SEED, checkpoint=run, defer_policy=True)

## Drive

The Actor's actual sampled action is highlighted. Five simulator frames follow
each policy decision. Use the controls to pause, change speed or review frames
already captured. Restart and replay never rerun the model.

In [3]:
view = LiveDriveView(drive)
view.show()

## Takeaway

At each decision the Actor produces five probabilities, then one action is
sampled and executed. Probabilities can change with the observed state;
the driving seed changes traffic and sampling. Probabilities alone do not
prove good driving. Lab 4 trains a policy that Lab 3 can use without training.

---
The default C25K agent was trained for 24,996 environment steps (999 optimizer
updates) with a reward for driving fast and −40 for a collision. A selected
Lab 4 run uses its own training and reward settings. The colored
road is a display for people; the Actor receives only the 5×5 table.